# 第3回 両替予算の計算と2つのドル・円レート


前半は第2回の続きで、2024年のドル・円レート（日銀、東京17時）を使う。値の比較、`if` による条件分岐、リストと繰り返し、折れ線グラフ、平均・最小・最大を順に練習し、演習1では12万円で買えるドルを計算する。

後半では日本銀行とFRB（米国の中央銀行）がそれぞれ公表しているドル・円レートを比べる。東京とニューヨークでは値を記録する時刻が違うので、同じ日付でも値は少しずれる。どちらの国が休みの日に値がないのかを確かめ、2つの値の差が大きくなる日を探す。


## 実行の準備

`finance_data.zip` をNotebookと同じフォルダに置き、下のセルを実行する。セルは上から順に実行すること。「変数が見つからない」というエラーが出たら、前のセルを実行し忘れている。

In [ ]:
from pathlib import Path
from zipfile import ZipFile

if not Path("course_data.py").exists():
    with ZipFile("finance_data.zip") as archive:
        archive.extractall(".")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from course_data import (
    load_boj_fx, load_boj_monthly, load_fed_fx, load_fed_rates,
    configure_plotting, format_axis,
)

configure_plotting()
pd.options.display.float_format = "{:.3f}".format


## 第2回で作った変数

新しいNotebookには第2回の変数が残っていないので、次のセルで作り直す。`fx` は日銀のドル・円の表、`rate` はその列から欠測の日を除いたSeries、`rate_2024` は2024年の部分である。`first` と `last` は2024年の最初と最後の観測、`change` はその差（円／ドル）を表す。

円の予算 $B$ で買えるドルの量は、$B$ をレート $S$（円／ドル）で割った $Q=B/S$ である。演習1でもこの式を使う。


In [ ]:
fx = load_boj_fx()
rate = fx["usd_jpy"].dropna()
rate_2024 = rate.loc["2024"]

first = rate_2024.iloc[0]
last = rate_2024.iloc[-1]
change = last - first

print(f"2024年の観測数：{len(rate_2024):,}")
print(f"最初の観測：{rate_2024.index[0].date()}、{first:.2f} 円／ドル")
print(f"最後の観測：{rate_2024.index[-1].date()}、{last:.2f} 円／ドル")
print(f"変化額：{change:.2f} 円／ドル")


## データ型と比較

Pythonでは、数と文字を別の種類（データ型）として扱う。`type()` で、値がどの種類かを調べられる。


In [ ]:
label = "ドル・円"
print(type(label))
print(type(first))


`str` は文字列（文字の並び）、`numpy.float64` は小数を含む数を表す。`first` は数なので計算に使えるが、文字列の `"ドル・円"` は割り算などの計算には使えない。


次に、2つの数を比べる。`>` や `<` で比べると、結果は `True`（正しい）か `False`（正しくない）になる。


In [ ]:
print(last > first)
print(last < first)


最後のレート（157.89円）は最初のレート（143.38円）より大きいので、`last > first` は `True`、`last < first` は `False` になった。


`if` を使うと、条件によって実行する処理を変えられる。

- `if 条件:` の条件が `True` なら、その下の字下げした行を実行する。
- `elif 条件:` は、上の条件が `False` だったときに、次の条件を調べる。
- `else:` は、どの条件にも当てはまらなかったときに実行する。

行末の `:` と、次の行の字下げ（先頭の空白）が、どこまでがその処理かを示す。


In [ ]:
if change > 0:
    print("2024年の最初と最後の観測を比べると、円安・ドル高です。")
elif change < 0:
    print("2024年の最初と最後の観測を比べると、円高・ドル安です。")
else:
    print("2024年の最初と最後の観測値は同じです。")


`change` は14.51でプラスなので、最初の `print` だけが実行された。

値を変えると、実行される行も変わる。下のセルでは、変化額が −3 円だった場合を `test_change` という別の名前で試す。


In [ ]:
test_change = -3

if test_change > 0:
    print("円安・ドル高です。")
elif test_change < 0:
    print("円高・ドル安です。")
else:
    print("変化はありません。")


今度は2つ目の `print` が実行された。`-3` を `0` に書き換えて実行し、`else` の行が実行されることも確かめてみよう。


## リストと繰り返し

いくつかの値を順に並べたものをリストという。まず2024年のレートから、先頭の3件を取り出す。`.iloc[:3]` は「先頭から3件」を表す。


In [ ]:
rate_2024.iloc[:3]


日付とレートが3件並んだ。これを `.tolist()` でリストに変え、`values` という名前を付ける。


In [ ]:
values = rate_2024.iloc[:3].tolist()
values


`[ ]` の中に、3つのレートがカンマで区切られて並んだ。リストにすると日付はなくなり、値だけが残る。

リストの件数は `len()` で数え、何番目の値かは `values[番号]` で取り出す。Pythonでは先頭を0番目と数えるので、先頭の値は `values[0]`、その次の値は `values[1]` になる。


In [ ]:
print(len(values))
print(values[0])
print(values[1])


件数は3、0番目は143.38、1番目は145.07である。


`for value in values:` は、リストの値を1つずつ `value` に入れて、その下の字下げした行を繰り返し実行する。


In [ ]:
for value in values:
    print(f"1ドル = {value:.2f} 円なら、1,000ドルの支払いは {1_000 * value:,.0f} 円")


3つの値について、同じ `print` が3回実行された。リストを作るセルの `.iloc[:3]` を `.iloc[:5]` に変えて、そのセルとこのセルを実行し直してみよう。今度は5行表示される。


## レートの推移

2024年のレートを折れ線グラフにする。グラフを描くコードは、次の部分からなる。

- `fig, ax = plt.subplots(figsize=(9, 4))` で図を用意する。`fig` は図全体、`ax` は線を描く場所で、`figsize` は図の横と縦の大きさである。
- `ax.plot(横軸, 縦軸)` で線を描く。横軸は日付（`rate_2024.index`）、縦軸はレート（`rate_2024`）で、`linewidth=1` は線の太さである。
- `ax.set()` で、題名（`title`）と縦軸の名前（`ylabel`）を付ける。
- `format_axis()` は目盛を見やすく整える。値は変えない。

最後の `fig.tight_layout()` は余白を整え、`plt.show()` で図を表示する。


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(rate_2024.index, rate_2024, linewidth=1)
ax.set(title="東京市場のドル・円（2024年）", ylabel="円／ドル")
format_axis(ax)
fig.tight_layout()
plt.show()


1月の143円台から円安が進み、7月初めには161円台まで上がった。その後は円高に戻って9月には140円台まで下がり、年末は157円台になっている。1年の間に20円ほどの幅で動いたことが分かる。


## 期間の代表値

`.mean()`、`.min()`、`.max()` で平均・最小・最大を求める。平均は観測のある各日を同じ重みで平均したもの。

In [ ]:
print(f"平均：{rate_2024.mean():.2f} 円／ドル")
print(f"最小：{rate_2024.min():.2f} 円／ドル")
print(f"最大：{rate_2024.max():.2f} 円／ドル")


最小や最大になった日付は、`.idxmin()` と `.idxmax()` で分かる。2024年は演習1で使うので、ここでは2023年のレートで試す。まず2023年の部分を `rate_2023` とし、最小になった日を調べる。


In [ ]:
rate_2023 = rate.loc["2023"]
rate_2023.idxmin()


`Timestamp('2023-01-16 00:00:00')` と表示された。`Timestamp` は日付と時刻を表す型で、ここでは2023年1月16日を指している。`.date()` を付けると日付だけになる。最大になった日と合わせて表示する。


In [ ]:
print(f"最小：{rate_2023.min():.2f} 円／ドル（{rate_2023.idxmin().date()}）")
print(f"最大：{rate_2023.max():.2f} 円／ドル（{rate_2023.idxmax().date()}）")


2023年は1月16日に最も円高（127.99円）、11月13日に最も円安（151.74円）だった。`.min()` と `.max()` は値そのものを、`.idxmin()` と `.idxmax()` はその値になった日付を返す。


## 円の予算と購入できるドル

たとえば15,000円で100ドル以上を買いたいとする。上限は 15,000 ÷ 100 ＝ 150円／ドルで、150円以下の日が条件に合う。

文字で書くと、予算 $B$ で $Q^*$ ドル以上を買えるのは、レートが上限以下の日である（$B,Q^*,S>0$）。

$$\frac{B}{S}\geq Q^* \quad\Longleftrightarrow\quad S\leq\frac{B}{Q^*}$$


## 条件に合う日の選び方

演習1では、条件に合う日だけを取り出す。そのときに使う `.loc` を、例で確かめておく。

`.loc[ ]` は、`[ ]` の中で指定した行を取り出す。第2回では日付を入れて期間を選んだ。


In [ ]:
rate_2024.loc["2024-07-01":"2024-07-05"]


`[ ]` の中には、日付の代わりに `True`／`False` を入れることもできる。まず、レートが160円以上かどうかを日ごとに調べる。比較の結果は、日付ごとの `True`／`False` になる。


In [ ]:
high = rate_2024 >= 160
high.loc["2024-06-24":"2024-07-12"]


6月27日から7月11日まで `True` が並んでいる。この期間はレートが160円以上だった。


この `high` を `.loc[ ]` に入れると、`True` の日だけが残る。


In [ ]:
rate_2024.loc[high]


160円以上だった日の日付とレートだけが並んだ。何日あったかは `len()` で数えられる。


In [ ]:
print(f"160円以上の日：{len(rate_2024.loc[high])} 日")


2行をまとめて `rate_2024.loc[rate_2024 >= 160]` と書いても同じ結果になる。演習1の追加問題では、`160` の代わりに予算から計算した上限を使い、`>=` を `<=` に変える。


## 演習1：12万円の両替予算

2024年の各観測日に、12万円を一度にドルへ換えるとする。

1. 最も多くのドルを買える日と最も少なくなる日を求め、それぞれの日付・為替レート・購入できるドルを示せ。
2. 800ドルを買うために、円／ドルのレートはいくら以下である必要があるか。予算と目標額から計算せよ。

追加問題：その条件に合う観測を選び、日数を数えよ。全245観測日のうち何%かも求め、この割合を「翌年に800ドルを買える確率」と呼べるか考えよ。

休日・欠測日のレートは補わない。

In [ ]:
budget_ex1 = 120_000
goal_usd_ex1 = 800
# rate_2024.min(), .max(), .idxmin(), .idxmax() を使う。
# 予算内で800ドルを買えるレートの上限を計算する。
# 条件に合う観測を .loc[...] で選び、len(...) で数える。


### 演習1の記入欄

- 最も多く買える日・レート・ドルの金額：
- 最も少なくなる日・レート・ドルの金額：
- 800ドルを買えるレートの上限：
- 追加問題：条件に合う日数・割合と、翌年について判断するために不足していること：


<details>
<summary>演習1のヒント</summary>

レートが低い日ほど多く買える。`budget_ex1 / rate_2024.min()` のように計算する。条件に合う観測は `limit_ex1 = budget_ex1 / goal_usd_ex1` として `rate_2024.loc[rate_2024 <= limit_ex1]` で選ぶ。割合を将来の確率とみなすには、将来の為替の動きについての仮定が要る。

</details>

## 2つのドル・円レート

ドル・円のレートは日本銀行（日銀）とFRBがそれぞれ公表している。どちらも「1ドル何円か」を表す数字だが、記録する場所と時刻が違う。

- 日銀：東京の17時の値
- FRB：ニューヨークの正午（12時）の値


まず日銀の表を読み込む。前半の `fx` と同じ表だが、FRBの表と区別しやすいように `boj` という名前を付ける。


In [ ]:
boj = load_boj_fx()
boj.head()


2000年1月1日から1日1行で並んでいる。1月1日〜3日は `NaN`（値なし）で、1日と2日は土日、3日は年始の休みにあたる。日銀の表には土日や休みの日にも行がある。


次にFRBの表を読み込み、`fed` という名前を付ける。


In [ ]:
fed = load_fed_fx()
fed.head()


FRBの表は2000年1月3日（月曜日）から始まり、土日の行はない。円のレートは `usd_jpy` の列にある。`eur_usd` と `gbp_usd` はユーロとポンドのレートで、今回は使わない。

1月3日は日本では休みだったが、米国では値が記録されている。


## 日本と米国の休日

2つの表の行数を比べる。`len()` は表の行数を返す。


In [ ]:
print(f"日銀の表：{len(boj):,} 行")
print(f"FRBの表：{len(fed):,} 行")


同じ2000〜2025年の表なのに、日銀の表のほうが行が多い。日銀の表には土日の行もあるからである。


次に値がない日（欠測）を数える。`.isna()` は値がない行に `True`、値がある行に `False` を付ける。最初の5行で確かめる。


In [ ]:
boj["usd_jpy"].isna().head()


値がない1月1日〜3日が `True` になっている。`.sum()` で足し合わせると `True` は1、`False` は0として数えられるので、値がない日の数が分かる。


In [ ]:
print(f"日銀の欠測：{boj['usd_jpy'].isna().sum():,} 日")
print(f"FRBの欠測：{fed['usd_jpy'].isna().sum():,} 日")


日銀の欠測が多いのは、土日の行がすべて欠測として数えられるからである。FRBの表には土日の行がないので、欠測は平日の休みの日だけになる。


## 2つの表を横に並べる

日本と米国で休みの日がどう違うかを、2024年のゴールデンウィークで見てみる。

まずそれぞれの表からドル・円の列を取り出す。どちらの列も `usd_jpy` という同じ名前なので、`.rename()` で「東京17時」「NY正午」という名前に付け替えておく。


In [ ]:
tokyo = boj["usd_jpy"].rename("東京17時")
ny = fed["usd_jpy"].rename("NY正午")

tokyo.loc["2024-04-26":"2024-05-08"]


東京の値には土日（4月27日・28日、5月4日・5日）の行もあり、その日は `NaN` になっている。4月29日、5月3日、5月6日も `NaN` である。


In [ ]:
ny.loc["2024-04-26":"2024-05-08"]


ニューヨークの値には土日の行がない。その代わり東京で `NaN` だった4月29日、5月3日、5月6日にも値がある。


2つを日付で横に並べて、1つの表にする。`pd.concat()` は `[ ]` の中に並べたSeriesや表をつなげる関数である。`axis=1` を付けると横につながる（付けないと縦に積み重なる）。

横に並べるときは日付が同じ行どうしがそろう。片方にしかない日付の行では、もう片方の列に `NaN` が入る。


In [ ]:
comparison = pd.concat([tokyo, ny], axis=1)
comparison.loc["2024-04-26":"2024-05-08"]


4月29日（昭和の日）、5月3日（憲法記念日）、5月6日（振替休日）は日本の祝日なので、東京だけが `NaN` になっている。土日は両方とも `NaN` である。

7月の初めも見てみる。


In [ ]:
comparison.loc["2024-07-01":"2024-07-08"]


今度は7月4日にニューヨークだけが `NaN` になっている。7月4日は米国の独立記念日である。値がない日は国ごとの休みによって違うことが分かる。

値がない日を0で埋めると「1ドル0円」というありえない値が入ってしまう。値がない日は `NaN` のままにしておく。


## 東京17時とニューヨーク正午

両方に値がある日でも、同じ行の2つの値は記録した時刻がずれている。時刻の順に並べると次のようになる。

| 出来事 | 日本時間 | ニューヨーク時間 |
|---|---|---|
| 日銀が値を記録（東京17時） | 当日 17:00 | 当日 4:00（3:00） |
| 米国の雇用統計・CPIの発表 | 当日 21:30（22:30） | 当日 8:30 |
| FRBが値を記録（ニューヨーク正午） | 翌日 1:00（2:00） | 当日 12:00 |

括弧の中は米国が冬時間の時期の時刻である（2024年の夏時間は3月10日〜11月3日）。ニューヨークの正午は東京の17時の8時間後（冬は9時間後）にあたる。

雇用統計は働く人の数や失業率を、CPI（消費者物価指数）は物価の動きを表す米国の統計で、米労働統計局（BLS）が毎月発表する。為替の取引をする人が注目する発表で、東京の17時より後、ニューヨークの正午より前に出る。


## 2つのレートの差

2024年について、2つのレートの差を計算する。差を出せるのは両方に値がある日だけなので、`.dropna()` で `NaN` を含む行を除く。表に `.dropna()` を使うと、どれか1つの列でも `NaN` がある行が消える。


In [ ]:
both = comparison.loc["2024"].dropna()

print(f"2024年の表の行数：{len(comparison.loc['2024'])} 行")
print(f"両方に値がある日：{len(both)} 日")
both.head()


列どうしの引き算は同じ行（同じ日付）の値どうしで計算される。「NY正午 − 東京17時」がプラスなら、ニューヨークの値のほうが円安・ドル高だったことになる。


In [ ]:
difference = both["NY正午"] - both["東京17時"]
difference.head()


差の平均・最小・最大を見る。


In [ ]:
print(f"差の平均：{difference.mean():.2f} 円／ドル")
print(f"差の最小：{difference.min():.2f} 円／ドル")
print(f"差の最大：{difference.max():.2f} 円／ドル")


平均はほぼ0で、2つのレートはふだんはほとんど同じである。それでも最も離れた日には3円ほどの差がある。


2つのレートと差を、上下2段の図に描く。`plt.subplots(2, 1)` で上下2段の図を作り、`axes[0]` に上の段、`axes[1]` に下の段を描く。

- `label=` で線に名前を付け、`.legend()` でその名前を表示する。
- `alpha=0.8` は線を少し透けさせる指定で、重なった線を見やすくする。
- `.axhline(0)` は高さ0の位置に横線を引く。


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(9, 6))
axes[0].plot(both.index, both["東京17時"], label="東京17時", linewidth=1)
axes[0].plot(both.index, both["NY正午"], label="NY正午", linewidth=1, alpha=0.8)
axes[0].set(ylabel="円／ドル", title="同じ日付のドル円")
axes[0].legend()
axes[1].plot(difference.index, difference, linewidth=0.9)
axes[1].axhline(0, color="gray", linewidth=0.8)
axes[1].set(ylabel="NY − 東京（円／ドル）")
for ax in axes:
    format_axis(ax)
fig.tight_layout()
plt.show()


上の段の2本の線はほとんど重なっている。下の段の差は0の上下を行き来し、約9割の日は1円以内に収まる。ところどころに2〜3円離れた日がある。


## 差が大きい日

差が大きかった日を探す。円安側と円高側を同じように比べたいので、マイナスの符号を外した「差の大きさ」（絶対値）を使う。`.abs()` は絶対値を返す。

`pd.DataFrame({ })` は `"列の名前": データ` の組を並べて表を作る。差と差の大きさの2列を持つ表を作り、`gap` と名付ける。


In [ ]:
gap = pd.DataFrame({
    "NY − 東京": difference,
    "差の大きさ": difference.abs(),
})
gap.head()


`.sort_values("差の大きさ", ascending=False)` で、差の大きさが大きい順に並べ替える。`ascending=False` は「大きい順」の指定で、付けないと小さい順になる。上から10日を表示する。


In [ ]:
gap.sort_values("差の大きさ", ascending=False).head(10)


1位は7月11日で、ニューヨークのほうが3円以上円高だった。これらの日に何があったのかを、米国の雇用統計とCPIの発表日と照らし合わせてみる。


BLSが公表している2024年の発表日をリストにする。`pd.to_datetime()` は `"2024-01-05"` のような文字列を日付に変える関数である。


In [ ]:
jobs_2024 = pd.to_datetime([
    "2024-01-05", "2024-02-02", "2024-03-08", "2024-04-05", "2024-05-03", "2024-06-07",
    "2024-07-05", "2024-08-02", "2024-09-06", "2024-10-04", "2024-11-01", "2024-12-06",
])
cpi_2024 = pd.to_datetime([
    "2024-01-11", "2024-02-13", "2024-03-12", "2024-04-10", "2024-05-15", "2024-06-12",
    "2024-07-11", "2024-08-14", "2024-09-11", "2024-10-10", "2024-11-13", "2024-12-11",
])
print(f"雇用統計：{len(jobs_2024)} 回、CPI：{len(cpi_2024)} 回")


`gap.index.isin(jobs_2024)` は、`gap` の各行の日付が `jobs_2024` に入っていれば `True`、入っていなければ `False` を返す。

この結果を新しい列として `gap` に加える。`gap["雇用統計"] = ...` のように、まだない列の名前に代入すると列が増える。CPIも同じように加えて、差が大きい10日を表示し直す。


In [ ]:
gap["雇用統計"] = gap.index.isin(jobs_2024)
gap["CPI"] = gap.index.isin(cpi_2024)

gap.sort_values("差の大きさ", ascending=False).head(10)


差が大きい10日のうち5日が、雇用統計かCPIの発表日だった。発表日は両方に値がある236日のうち23日しかない（5月3日の雇用統計の日は日本の祝日で、東京の値がない）。1割に満たない発表日が上位10日の半分を占めている。


図でも確かめる。`|` は「または」を表す記号で、`release` は雇用統計かCPIのどちらかの発表日に `True` になる。


In [ ]:
release = gap["雇用統計"] | gap["CPI"]
print(f"発表日：{release.sum()} 日")


差の線の上に、発表日を赤い点で重ねる。`ax.scatter()` は点を描く関数である。`gap.index[release]` と `gap.loc[release, "NY − 東京"]` で、発表日の日付と差だけを取り出して渡す。`zorder=3` は点を線より手前に描く指定である。


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(gap.index, gap["NY − 東京"], color="gray", linewidth=0.9)
ax.scatter(gap.index[release], gap.loc[release, "NY − 東京"],
           color="C3", zorder=3, label="雇用統計・CPIの発表日")
ax.axhline(0, color="gray", linewidth=0.8)
ax.set(title="NY正午と東京17時の差（2024年）", ylabel="NY − 東京（円／ドル）")
ax.legend(loc="lower right")
format_axis(ax)
fig.tight_layout()
plt.show()


差が大きく離れた日の半分ほどに、赤い点が重なっている。

発表は東京の17時より後に出るので、その影響はまずニューヨークの正午の値に入り、東京の値に表れるのは次の日になる。同じ日付の差が大きくなるのは、2つの時刻の間に大きなニュースがあった日である。


## 次の日の東京の動き

ニューヨークの正午は、その日の東京17時と次の日の東京17時の間にある。それならニューヨークで円安になっていた日には、次の日の東京も円安になっているのではないか。これを確かめる。

まず次の日の東京の値を同じ行に並べる。`rate.shift(-1)` は値を1つ上にずらして、各行に次の観測日の値を入れる。`rate` は前半で作った、値のある日だけの東京のレートである。


In [ ]:
pd.DataFrame({
    "当日の値": rate,
    "次の観測日の値": rate.shift(-1),
}).loc["2024-01-04":"2024-01-12"]


1月4日の行には1月5日の値（145.07）が「次の観測日の値」として入っている。1月5日（金曜日）の行には、土日と成人の日（1月8日）をとばして、1月9日の値が入っている。


「次の観測日の値 − 当日の値」で、次の日までに東京のレートがどれだけ動いたかが分かる。これを `gap` の新しい列として加える。


In [ ]:
tokyo_change = rate.shift(-1) - rate
gap["次の東京の変化"] = tokyo_change
gap.head()


1月4日〜10日は、「NY − 東京」と「次の東京の変化」のプラス・マイナスがそろっている。1月11日はそろっていない。1年分でそろう日がどれくらいあるかを数える。


「NYが東京より円安だったか」と「次の日の東京が円安になったか」を、それぞれ `True`／`False` にする。


In [ ]:
ny_weak = gap["NY − 東京"] > 0        # NYが東京より円安なら True
next_weak = gap["次の東京の変化"] > 0  # 次の日の東京が円安なら True
ny_weak.head()


`pd.crosstab()` は2つの `True`／`False` の組合せごとに日数を数えて表にする。


In [ ]:
table = pd.crosstab(ny_weak, next_weak)
table


行は `ny_weak`（NYが東京より円安か）、列は `next_weak`（次の日の東京が円安か）で、どちらも `False`、`True` の順に並んでいる。表の左上に出ている「NY − 東京」「次の東京の変化」は、もとの列の名前である。分かりやすいように、行と列の見出しを日本語に付け替える。


In [ ]:
table.index = ["NYが東京以下", "NYが東京より円安"]
table.columns = ["次の東京は円高・変化なし", "次の東京は円安"]
table


NYが東京より円安だった日（下の行）は、次の日の東京も円安になった日が多い。反対にNYが東京以下だった日（上の行）では、次の日の東京が円高か変化なしだった日のほうが多い。ニューヨークの正午の値には次の東京17時までの動きの一部がすでに入っているので、このように向きがそろいやすい。

この関係を使えるのは、ニューヨークの正午の値が出た後である。東京の17時の時点ではその日のニューヨークの値はまだ出ておらず、出るのは8〜9時間後になる。日付だけで並べた表には、後になって分かる値も同じ行に入ってしまう。先の値を予想するために表を作るときは、その時点で分かっていた値だけを使うように気をつけよう。


## 演習2：発表日の差と次の日の東京

1. 雇用統計かCPIの発表日と、それ以外の日に分けて、`差の大きさ` の列の平均を求めよ。発表日の平均はそれ以外の日の何倍か。
2. 表 `table` から、次の2つの割合を求めよ。
   - NYが東京より円安だった日のうち、次の日の東京も円安だった日の割合
   - NYが東京以下だった日のうち、次の日の東京が円高・変化なしだった日の割合
3. ある人が「ニューヨークの正午に東京17時より円安になっていたら、その日の東京17時のレートでドルを買っておけば得をする」と考えた。上の時刻の表を使って、この考えのどこに問題があるかを説明せよ。

追加問題：2023年について同じ表を作り、向きがそろった日の割合を2024年と比べよ。


In [ ]:
# 発表日は release（True／False）で選べる。~release はそれ以外の日を表す。
# gap.loc[release, "差の大きさ"].mean() のように平均を求める。
# table.loc["NYが東京より円安", "次の東京は円安"] で表の日数を取り出せる。


### 演習2の記入欄

- 発表日とそれ以外の日の差の大きさの平均、何倍か：
- 向きがそろった日の割合（2通り）：
- 東京17時のレートで買っておけば得をするという考えの問題点：
- 追加問題：2023年の割合と、2024年との比較：


<details>
<summary>演習2のヒント</summary>

`~` は `True` と `False` を入れ替えるので、`gap.loc[~release, "差の大きさ"]` で発表日以外の日を選べる。割合は、行の合計 `table.loc["NYが東京より円安"].sum()` で割る。問3では、ニューヨークの正午の値が分かる時刻と東京の17時を比べる。追加問題では `comparison.loc["2023"]` から差を作り直し、同じ手順で表を作る。

</details>


## 出典

[日本銀行「外国為替市況」](https://www.boj.or.jp/statistics/market/forex/fxdaily/index.htm)の東京市場ドル・円スポット、17時時点（円／ドル）を使用。配布データの対象期間は2000〜2025年である。

米国の為替レートは [FRB H.10（為替）](https://www.federalreserve.gov/releases/h10/)による。

米国の雇用統計と消費者物価指数の発表日・時刻は[BLS「Schedule of Selected Releases 2024」](https://www.bls.gov/schedule/2024/home.htm)による。
